# Exoplanet Life Sustainability Score (LSS) — Data Pipeline

## Project Goal
Build a **Life Sustainability Score (LSS)** for ~6,000 confirmed exoplanets from the NASA Exoplanet Archive, then prepare a model-ready dataset to predict LSS from observable planetary and stellar features.

## Pipeline Overview

| Step | Cell(s) | Task | Input → Output |
|------|---------|------|----------------|
| **1–2** | Cell 2 | Load raw CSV + select 15 columns | `PSCompPars_2026.csv` → `exoplanets_selected.csv` (6128 × 15) |
| **3** | Cell 3 | Missing value imputation (median + MICE) | → `exoplanets_step3_clean.csv` (6128 × 15, 0 missing) |
| **4** | Cell 4 | Outlier removal (domain + Z-score) | → `exoplanets_step4_clean.csv` (~5456 × 15) |
| **5** | Cells 5–6 | Log transforms + hotfix | → `exoplanets_step5_transformed.csv` (5456 × 15) |
| **6** | Cell 7 | Collinearity analysis (VIF) + feature dropping | → `exoplanets_step6_final_features.csv` (5456 × 10) |
| **7** | Cell 8 | LSS target construction (5 sub-scores) | → `exoplanets_step7_model_ready.csv` (5456 × 11) |
| **8** | Cell 9 | Stratified split + RobustScaler | → `X_train.csv`, `X_test.csv`, etc. (4364 / 1092) |

## Key Columns

- **Identifiers**: `pl_name`, `hostname`
- **Planetary**: `pl_rade`, `pl_bmasse`, `pl_dens`, `pl_orbeccen`, `pl_eqt`, `pl_insol`
- **Orbital**: `pl_orbper`, `pl_orbsmax`
- **Stellar**: `st_teff`, `st_lum`, `st_rad`, `st_mass`, `st_age`
- **Target**: `LSS` (0–1, weighted composite of 5 habitability sub-scores)

## Outputs
- 11 `.csv` artefacts (one per pipeline stage + train/test splits)
- 10 `.png` visualisations
- `robust_scaler.pkl` (fitted scaler for inference)

## Steps 1–2: Data Loading & Column Selection

**What this cell does:**
1. Loads the raw NASA Exoplanet Archive CSV (`PSCompPars_2026.03.05_02.02.54.csv`) — 6128 planets × 54 columns.
2. Saves a raw backup (`exoplanets_raw_backup.csv`).
3. Selects 15 columns: 2 identifiers + 6 planetary + 2 orbital + 5 stellar.
4. Builds a missing-value report with pre-assigned imputation actions.
5. Saves `exoplanets_selected.csv`.

**Decisions:**
- `pl_bmasse` (best mass estimate) used instead of `pl_masse`.
- All error-bar and limit-flag columns dropped — only central estimates retained.
- Missing rates range from 0.1 % (`st_mass`) to 29.8 % (`pl_insol`).

**Issues found:**
1. Steps 1 and 2 are in a **single cell** — splitting them would make each independently re-runnable.
2. The "Action" column in the missing-value report is **hardcoded**, not computed from the actual `% Missing` threshold. If the data changes, the labels won't auto-update.
3. File path is an **absolute path** — not portable to another machine.

In [6]:
# ============================================================
# STEP 1 — LOAD YOUR DOWNLOADED FILE & INITIAL INSPECTION
# ============================================================

import pandas as pd
import numpy as np

FILE_PATH = "/home/aaryajain/exoplanet_project/PSCompPars_2026.03.05_02.02.54.csv"

# NASA archive files have comment lines starting with #
df_raw = pd.read_csv(FILE_PATH, comment="#")
print(f"✅ Loaded successfully")
print(f"   Shape: {df_raw.shape[0]} planets × {df_raw.shape[1]} columns")

# Save raw backup
df_raw.to_csv("exoplanets_raw_backup.csv", index=False)
print(f"   Raw backup saved → exoplanets_raw_backup.csv")

# Quick inspection
print("\n--- FIRST 3 ROWS ---")
print(df_raw.head(3))

print("\n--- BASIC STATS ---")
print(df_raw.describe())


# ============================================================
# STEP 2 — COLUMN SELECTION
# ============================================================

# NOTE: Archive uses pl_bmasse (best mass estimate), NOT pl_masse
ID_COLS        = ["pl_name", "hostname"]
PLANETARY_COLS = ["pl_rade", "pl_bmasse", "pl_dens", "pl_orbeccen", "pl_eqt", "pl_insol"]
ORBITAL_COLS   = ["pl_orbper", "pl_orbsmax"]
STELLAR_COLS   = ["st_teff", "st_lum", "st_rad", "st_mass", "st_age"]

ALL_KEEP = ID_COLS + PLANETARY_COLS + ORBITAL_COLS + STELLAR_COLS

# Verify all columns exist
missing_cols   = [c for c in ALL_KEEP if c not in df_raw.columns]
available_cols = [c for c in ALL_KEEP if c in df_raw.columns]

if missing_cols:
    print(f"\n⚠️  Columns NOT found: {missing_cols}")
else:
    print(f"\n✅ All {len(ALL_KEEP)} requested columns found")

# Select columns
df = df_raw[available_cols].copy()
print(f"   Columns kept : {df.shape[1]}  (dropped {df_raw.shape[1] - df.shape[1]})")
print(f"   Planets kept : {df.shape[0]}")

# ---- Missing value report ----
print("\n--- MISSING VALUE REPORT ---")
missing_report = pd.DataFrame({
    "Column"       : df.columns,
    "Category"     : (["ID"]*2 + ["Planetary"]*6 + ["Orbital"]*2 + ["Stellar"]*5),
    "Non-Null"     : df.notna().sum().values,
    "Missing"      : df.isna().sum().values,
    "% Missing"    : (df.isna().mean() * 100).round(1).values,
    "Action"       : [
        "keep", "keep",                      # IDs
        "median impute",                     # pl_rade       0.8%
        "median impute",                     # pl_bmasse     0.5%
        "median impute",                     # pl_dens       2.3%
        "MICE impute",                       # pl_orbeccen  15.1%
        "MICE impute",                       # pl_eqt       25.3%
        "MICE impute",                       # pl_insol     29.8%
        "median impute",                     # pl_orbper     5.4%
        "median impute",                     # pl_orbsmax    5.2%
        "median impute",                     # st_teff       4.7%
        "median impute",                     # st_lum        4.9%
        "median impute",                     # st_rad        5.0%
        "median impute",                     # st_mass       0.1%
        "MICE impute",                       # st_age       21.2%
    ]
}).set_index("Column")

print(missing_report.to_string())

# Save selected dataset
df.to_csv("exoplanets_selected.csv", index=False)
print(f"\n✅ Selected dataset saved → exoplanets_selected.csv")
print(f"   Final shape: {df.shape[0]} planets × {df.shape[1]} columns")

✅ Loaded successfully
   Shape: 6128 planets × 54 columns
   Raw backup saved → exoplanets_raw_backup.csv

--- FIRST 3 ROWS ---
    pl_name hostname  pl_orbper  pl_orbpererr1  pl_orbpererr2  pl_orbperlim  \
0  11 Com b   11 Com  323.21000           0.06          -0.05           0.0   
1  11 UMi b   11 UMi  516.21997           3.20          -3.20           0.0   
2  14 And b   14 And  186.76000           0.11          -0.12           0.0   

   pl_orbsmax  pl_orbsmaxerr1  pl_orbsmaxerr2  pl_orbsmaxlim  ...  \
0       1.178            0.00            0.00            0.0  ...   
1       1.530            0.07           -0.07            0.0  ...   
2       0.775            0.00            0.00            0.0  ...   

   st_masserr2  st_masslim   st_lum  st_lumerr1  st_lumerr2  st_lumlim  \
0        -0.63         0.0  1.97823     0.18002    -0.15868        0.0   
1        -0.69         0.0  2.42951     0.00801    -0.00816        0.0   
2        -0.29         0.0  1.83992     0.08135    -0.04

## Step 3: Missing Value Imputation

**What this cell does:**
1. **Visualises** missingness with a bar chart (`missing_values_bar.png`) and heatmap (`missing_heatmap.png`).
2. **Log-transforms** 7 skewed features before imputation so MICE operates on near-normal distributions.
3. **Median imputation** (SimpleImputer) for 9 low-missingness columns (< 10 % missing).
4. **MICE imputation** (IterativeImputer with RandomForestRegressor) for 4 high-missingness columns: `pl_orbeccen` (15 %), `pl_eqt` (25 %), `pl_insol` (30 %), `st_age` (21 %).
5. Reverses the log transforms to restore original units.
6. Verifies zero missing values remain.
7. Plots before-vs-after distributions (`imputation_distributions.png`).
8. Saves `exoplanets_step3_clean.csv` (6128 × 15).

**Decisions:**
- Threshold: < 10 % → median; > 15 % → MICE.
- MICE estimator: `RandomForestRegressor(n_estimators=10)`, 10 iterations.
- Log1p used for shift-safe log transform on skewed columns.

**Issues found:**
1. **Cell currently has an execution error** (see output) — must be re-run after the kernel has scikit-learn available.
2. **MICE capacity is low**: only 10 trees and 10 iterations. Increasing to 50–100 trees and 15–20 iterations would yield more stable imputed values.
3. **No physical-bounds clipping after imputation** — MICE can generate out-of-range values (e.g., negative eccentricity). A post-imputation `clip()` to each feature's physical domain should be added.
4. **Imputation is fit on the full dataset**, not train-only. This introduces mild **data leakage** — ideally imputation would be fit on the training set and applied to the test set.  Since the target (LSS) is derived rather than observed, the impact is reduced but the practice is still not ideal.
5. **Imputation before outlier removal** (Step 4 comes next) — outliers in observed data influence the imputed values, then some of those imputed rows may themselves get removed as outliers.

In [16]:
# ============================================================
# STEP 3 — MISSING VALUE ANALYSIS & IMPUTATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer   # required before import
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# ---------- Load selected dataset from Step 2 ----------
df = pd.read_csv("exoplanets_selected.csv")
print(f"Loaded: {df.shape[0]} planets × {df.shape[1]} columns")

FEATURE_COLS = ["pl_rade", "pl_bmasse", "pl_dens", "pl_orbeccen", "pl_eqt",
                "pl_insol", "pl_orbper", "pl_orbsmax",
                "st_teff", "st_lum", "st_rad", "st_mass", "st_age"]


# ============================================================
# 3A — VISUALISE MISSING VALUES
# ============================================================

missing_pct = df[FEATURE_COLS].isna().mean() * 100

plt.figure(figsize=(10, 5))
bars = plt.barh(missing_pct.index, missing_pct.values,
                color=["#e74c3c" if v > 20 else "#f39c12" if v > 10 else "#2ecc71"
                       for v in missing_pct.values])
plt.axvline(20, color="red",    linestyle="--", linewidth=1, label="20% threshold (MICE)")
plt.axvline(10, color="orange", linestyle="--", linewidth=1, label="10% threshold")
plt.xlabel("% Missing")
plt.title("Missing Values per Feature")
plt.legend()
plt.tight_layout()
plt.savefig("missing_values_bar.png", dpi=150)
plt.close()
print("✅ Plot saved → missing_values_bar.png")

# Heatmap of missingness pattern
plt.figure(figsize=(14, 6))
sns.heatmap(df[FEATURE_COLS].isna().T, cbar=False,
            cmap=["#2ecc71", "#e74c3c"], yticklabels=True)
plt.title("Missingness Pattern (red = missing)")
plt.xlabel("Planet index")
plt.tight_layout()
plt.savefig("missing_heatmap.png", dpi=150)
plt.close()
print("✅ Plot saved → missing_heatmap.png")


# ============================================================
# 3B — LOG TRANSFORM SKEWED FEATURES BEFORE IMPUTATION
# ============================================================
# Skewness check showed: pl_orbper=76, pl_dens=40, pl_insol=18
# Log transform makes distributions normal → MICE works much better

LOG_COLS = ["pl_orbper", "pl_orbsmax", "pl_bmasse", "pl_dens",
            "pl_insol", "st_rad", "st_mass"]

df_work = df.copy()

for col in LOG_COLS:
    # Shift if any values <= 0 before log
    min_val = df_work[col].min()
    shift = abs(min_val) + 1e-6 if min_val <= 0 else 0
    df_work[f"{col}_log"] = np.log1p(df_work[col] + shift)

print("\n✅ Log-transformed columns:", LOG_COLS)

# Build working feature list (use log versions where applicable)
FEATURE_WORK = []
for col in FEATURE_COLS:
    FEATURE_WORK.append(f"{col}_log" if col in LOG_COLS else col)

print("   Working features:", FEATURE_WORK)


# ============================================================
# 3C — SIMPLE MEDIAN IMPUTATION (features < 10% missing)
# ============================================================

MEDIAN_COLS_ORIG = ["pl_rade", "pl_orbper", "pl_orbsmax",
                    "pl_bmasse", "pl_dens", "st_teff",
                    "st_lum", "st_rad", "st_mass"]

MEDIAN_COLS_WORK = [f"{c}_log" if c in LOG_COLS else c for c in MEDIAN_COLS_ORIG]

median_imputer = SimpleImputer(strategy="median")
df_work[MEDIAN_COLS_WORK] = median_imputer.fit_transform(df_work[MEDIAN_COLS_WORK])

print(f"\n✅ Median imputation done for {len(MEDIAN_COLS_WORK)} columns:")
for orig, work in zip(MEDIAN_COLS_ORIG, MEDIAN_COLS_WORK):
    print(f"   {orig:15s} → imputed via {work}")


# ============================================================
# 3D — MICE IMPUTATION (features > 15% missing)
# ============================================================
# Uses RandomForest internally — handles nonlinear relationships well
# pl_orbeccen (15%), pl_eqt (25%), pl_insol (30%), st_age (21%)

MICE_COLS_ORIG = ["pl_orbeccen", "pl_eqt", "pl_insol", "st_age"]
MICE_COLS_WORK = [f"{c}_log" if c in LOG_COLS else c for c in MICE_COLS_ORIG]

# MICE uses ALL features as predictors — pass full working feature set
mice_input = df_work[FEATURE_WORK].copy()

print("\n⏳ Running MICE imputation (this takes ~1-2 minutes)...")
mice_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=10, random_state=42),
    max_iter=10,
    random_state=42,
    verbose=1
)
mice_output = mice_imputer.fit_transform(mice_input)
mice_df = pd.DataFrame(mice_output, columns=FEATURE_WORK)

# Put MICE-imputed columns back into df_work
for col in MICE_COLS_WORK:
    df_work[col] = mice_df[col].values

print(f"\n✅ MICE imputation done for: {MICE_COLS_ORIG}")


# ============================================================
# 3E — REVERSE LOG TRANSFORM
# ============================================================

for col in LOG_COLS:
    log_col = f"{col}_log"
    # Reverse: expm1 undoes log1p
    df_work[col] = np.expm1(df_work[log_col])
    df_work.drop(columns=[log_col], inplace=True)

print("\n✅ Log transforms reversed — all features back in original units")


# ============================================================
# 3F — VERIFY — NO MISSING VALUES REMAIN
# ============================================================

remaining_missing = df_work[FEATURE_COLS].isna().sum()
print("\n--- POST-IMPUTATION MISSING CHECK ---")
print(remaining_missing)

if remaining_missing.sum() == 0:
    print("\n✅ All missing values handled — dataset is complete!")
else:
    print(f"\n⚠️  {remaining_missing.sum()} missing values remain — check above")


# ============================================================
# 3G — VISUAL CHECK: BEFORE vs AFTER DISTRIBUTIONS
# ============================================================

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

check_cols = ["pl_orbeccen", "pl_eqt", "pl_insol", "st_age",
              "pl_orbper", "pl_bmasse", "pl_dens", "st_rad"]

for i, col in enumerate(check_cols):
    ax = axes[i]
    # Before (original, drop NaN)
    ax.hist(df[col].dropna(), bins=40, alpha=0.5,
            color="#3498db", label="Before imputation", density=True)
    # After
    ax.hist(df_work[col], bins=40, alpha=0.5,
            color="#e74c3c", label="After imputation", density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")

plt.suptitle("Distribution Before vs After Imputation", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("imputation_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Distribution plot saved → imputation_distributions.png")


# ============================================================
# 3H — SAVE FINAL CLEAN DATASET
# ============================================================

df_clean = df_work[["pl_name", "hostname"] + FEATURE_COLS].copy()
df_clean.to_csv("exoplanets_step3_clean.csv", index=False)

print(f"\n✅ Clean dataset saved → exoplanets_step3_clean.csv")
print(f"   Final shape: {df_clean.shape[0]} planets × {df_clean.shape[1]} columns")
print(f"\n--- FINAL DATASET SUMMARY ---")
print(df_clean[FEATURE_COLS].describe().round(3))

Loaded: 6128 planets × 15 columns
✅ Plot saved → missing_values_bar.png
✅ Plot saved → missing_heatmap.png

✅ Log-transformed columns: ['pl_orbper', 'pl_orbsmax', 'pl_bmasse', 'pl_dens', 'pl_insol', 'st_rad', 'st_mass']
   Working features: ['pl_rade', 'pl_bmasse_log', 'pl_dens_log', 'pl_orbeccen', 'pl_eqt', 'pl_insol_log', 'pl_orbper_log', 'pl_orbsmax_log', 'st_teff', 'st_lum', 'st_rad_log', 'st_mass_log', 'st_age']

✅ Median imputation done for 9 columns:
   pl_rade         → imputed via pl_rade
   pl_orbper       → imputed via pl_orbper_log
   pl_orbsmax      → imputed via pl_orbsmax_log
   pl_bmasse       → imputed via pl_bmasse_log
   pl_dens         → imputed via pl_dens_log
   st_teff         → imputed via st_teff
   st_lum          → imputed via st_lum
   st_rad          → imputed via st_rad_log
   st_mass         → imputed via st_mass_log

⏳ Running MICE imputation (this takes ~1-2 minutes)...
[IterativeImputer] Completing matrix with shape (6128, 13)
[IterativeImputer] Change

/home/aaryajain/exoplanet_project/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


✅ Distribution plot saved → imputation_distributions.png

✅ Clean dataset saved → exoplanets_step3_clean.csv
   Final shape: 6128 planets × 15 columns

--- FINAL DATASET SUMMARY ---
        pl_rade  pl_bmasse   pl_dens  pl_orbeccen    pl_eqt   pl_insol  \
count  6128.000   6128.000  6128.000     6128.000  6128.000   6128.000   
mean      5.811    403.548     4.861        0.090   836.965    341.295   
std       5.426   1140.237    34.221        0.155   456.903   1136.220   
min       0.310      0.020     0.005        0.000    34.000      0.000   
25%       1.830      4.198     1.330        0.000   518.841     17.302   
50%       2.840      9.220     2.560        0.000   728.000     59.115   
75%      11.900    188.235     4.540        0.120  1074.460    264.383   
max      87.206   9534.852  2000.000        0.950  4050.000  44900.000   

          pl_orbper  pl_orbsmax    st_teff    st_lum    st_rad   st_mass  \
count  6.128000e+03    6128.000   6128.000  6128.000  6128.000  6128.000   

## Step 4: Outlier Detection & Removal

**What this cell does:**
1. Creates **boxplots before** removal (`boxplots_before.png`).
2. Applies **domain-knowledge cuts** — scientifically motivated min/max for each feature (e.g., planet mass < 13 M_Jup, stellar age < 14 Gyr).
3. Applies **Z-score outlier removal** (|z| > 3.5 on log-transformed values) for statistical outliers that passed domain cuts.
4. Logs every removal decision (feature, rule, count, reason).
5. Creates **boxplots after** removal and a before-vs-after histogram overlay (`boxplots_after.png`, `outlier_distributions.png`).
6. Saves `exoplanets_step4_clean.csv` (~5456 × 15).

**Decisions:**
- Domain thresholds are all justified with astrophysical reasoning (documented inline).
- Z-score threshold 3.5 (conservative) rather than 3.0 — intentionally preserves rare but real planets.
- Log-transform applied to 7 skewed columns before computing Z-scores.

**Issues found:**
1. **Sequential domain cuts**: each rule filters *after* the previous, so counts per rule depend on order. The total removed is the same, but per-feature counts would differ if the order changed. To get accurate per-feature counts, apply all masks independently first, then combine before filtering.
2. **Sequential Z-score removal**: similarly, each column's Z-scores are recalculated on the progressively shrunk dataset. Removing extremes in column A changes the Z-scores for column B. A simultaneous mask (flag all z-outliers, then remove once) would be more robust.
3. **No record of which specific planets were removed** — `removal_log` records counts but not planet names. Saving the removed rows to a CSV would aid debugging.

In [17]:
# ============================================================
# STEP 4 — OUTLIER DETECTION & REMOVAL
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- Load clean dataset from Step 3 ----------
df = pd.read_csv("exoplanets_step3_clean.csv")
print(f"Loaded: {df.shape[0]} planets × {df.shape[1]} columns")

FEATURE_COLS = ["pl_rade", "pl_bmasse", "pl_dens", "pl_orbeccen", "pl_eqt",
                "pl_insol", "pl_orbper", "pl_orbsmax",
                "st_teff", "st_lum", "st_rad", "st_mass", "st_age"]

df_clean = df.copy()
removal_log = []   # track every removal decision


# ============================================================
# 4A — VISUALISE RAW DISTRIBUTIONS (BEFORE)
# ============================================================

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    axes[i].boxplot(df_clean[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor="#3498db", alpha=0.6))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel("Value")

# Hide unused subplot
axes[-1].set_visible(False) if len(FEATURE_COLS) < len(axes) else None

plt.suptitle("Boxplots BEFORE Outlier Removal", fontsize=14)
plt.tight_layout()
plt.savefig("boxplots_before.png", dpi=150)
plt.close()
print("✅ Saved → boxplots_before.png")


# ============================================================
# 4B — DOMAIN-KNOWLEDGE THRESHOLDS
# ============================================================
# These are NOT arbitrary — each cut has a scientific basis.

domain_rules = {
    # Feature         : (min,    max,    reason)
    "pl_bmasse"  : (0.1,   4131.0,  "Brown dwarf boundary = 13 Jupiter masses = 4131 Earth masses"),
    "pl_rade"    : (0.3,   25.0,    "Below sub-Earth; above ~2.5x Jupiter radius = unphysical"),
    "pl_dens"    : (0.001, 100.0,   "Above 100 g/cm³ is physically implausible for planets"),
    "pl_orbper"  : (0.1,   100000,  "Periods >100,000 days = wide-separation imaged companions"),
    "pl_orbsmax" : (0.001, 100.0,   "Semi-major axis >100 AU = not classic orbital planet"),
    "pl_eqt"     : (50,    4000.0,  "Below 50K unphysical; above 4000K = ultra-hot Jupiters limit"),
    "pl_insol"   : (0.0,   30000.0, "Above 30000 Earth flux = extreme irradiation outliers"),
    "pl_orbeccen": (0.0,   0.95,    "Eccentricity must be 0-1; >0.95 is extreme/unphysical"),
    "st_teff"    : (2000,  40000.0, "Below 2000K = brown dwarf host; above 40000K = white dwarf"),
    "st_rad"     : (0.01,  50.0,    "Above 50 solar radii = AGB/supergiant, not main sequence"),
    "st_mass"    : (0.05,  8.0,     "Above 8 solar masses = O-type star, not habitable context"),
    "st_age"     : (0.0,   14.0,    "Above 14 Gyr = older than universe"),
    "st_lum"     : (-6.0,  3.8,     "Retain full observed range from archive"),
}

print("\n=== DOMAIN-BASED REMOVAL ===")
print(f"{'Feature':<15} {'Rule':<25} {'Removed':>8}  Reason")
print("-" * 90)

for col, (lo, hi, reason) in domain_rules.items():
    if col not in df_clean.columns:
        continue
    mask_out = (df_clean[col] < lo) | (df_clean[col] > hi)
    n_removed = mask_out.sum()
    removal_log.append({
        "step": "domain",
        "feature": col,
        "rule": f"[{lo}, {hi}]",
        "removed": n_removed,
        "reason": reason
    })
    df_clean = df_clean[~mask_out].copy()
    print(f"{col:<15} [{lo}, {hi}]{'':>5} {n_removed:>6} rows   {reason}")

print(f"\nAfter domain cuts: {df_clean.shape[0]} planets remain "
      f"(removed {df.shape[0] - df_clean.shape[0]} total)")


# ============================================================
# 4C — STATISTICAL OUTLIER DETECTION (Z-SCORE ON LOG DATA)
# ============================================================
# Raw Z-score fails on skewed data → log-transform first, then Z-score
# Threshold: |z| > 3.5 (more conservative than 3.0 to preserve rare planets)

LOG_COLS = ["pl_orbper", "pl_orbsmax", "pl_bmasse", "pl_dens",
            "pl_insol", "st_rad", "st_mass"]

Z_THRESHOLD = 3.5

print(f"\n=== Z-SCORE OUTLIER REMOVAL (threshold = ±{Z_THRESHOLD}) ===")
print(f"{'Feature':<15} {'Removed':>8}  {'Remaining':>10}")
print("-" * 40)

before_z = df_clean.shape[0]

for col in FEATURE_COLS:
    if col not in df_clean.columns:
        continue

    # Log-transform if skewed
    vals = np.log1p(df_clean[col]) if col in LOG_COLS else df_clean[col]

    # Compute Z-score
    z_scores = (vals - vals.mean()) / vals.std()
    mask_out = z_scores.abs() > Z_THRESHOLD
    n_removed = mask_out.sum()

    if n_removed > 0:
        removal_log.append({
            "step": "zscore",
            "feature": col,
            "rule": f"|z| > {Z_THRESHOLD}",
            "removed": n_removed,
            "reason": "Statistical outlier on log-transformed data"
        })
        df_clean = df_clean[~mask_out].copy()
        print(f"{col:<15} {n_removed:>8}  {df_clean.shape[0]:>10}")

print(f"\nAfter Z-score cuts: {df_clean.shape[0]} planets remain "
      f"(removed {before_z - df_clean.shape[0]} additional)")


# ============================================================
# 4D — FULL REMOVAL SUMMARY
# ============================================================

total_removed = df.shape[0] - df_clean.shape[0]
pct_removed   = total_removed / df.shape[0] * 100

print("\n=== FULL REMOVAL SUMMARY ===")
print(f"  Original rows  : {df.shape[0]}")
print(f"  Rows removed   : {total_removed} ({pct_removed:.1f}%)")
print(f"  Rows remaining : {df_clean.shape[0]}")

removal_df = pd.DataFrame(removal_log)
print("\nDetailed log:")
print(removal_df.to_string(index=False))


# ============================================================
# 4E — VISUALISE AFTER (BOXPLOTS + HISTOGRAM OVERLAY)
# ============================================================

# Boxplots after
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    axes[i].boxplot(df_clean[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor="#2ecc71", alpha=0.6))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel("Value")

axes[-1].set_visible(False) if len(FEATURE_COLS) < len(axes) else None
plt.suptitle("Boxplots AFTER Outlier Removal", fontsize=14)
plt.tight_layout()
plt.savefig("boxplots_after.png", dpi=150)
plt.close()
print("\n✅ Saved → boxplots_after.png")

# Before vs after histogram comparison
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    vals_before = np.log1p(df[col])     if col in LOG_COLS else df[col]
    vals_after  = np.log1p(df_clean[col]) if col in LOG_COLS else df_clean[col]

    ax.hist(vals_before, bins=50, alpha=0.4, color="#e74c3c",
            density=True, label="Before")
    ax.hist(vals_after,  bins=50, alpha=0.6, color="#2ecc71",
            density=True, label="After")
    label = f"log({col})" if col in LOG_COLS else col
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=7)

axes[-1].set_visible(False) if len(FEATURE_COLS) < len(axes) else None
plt.suptitle("Distributions Before vs After Outlier Removal", fontsize=14)
plt.tight_layout()
plt.savefig("outlier_distributions.png", dpi=150)
plt.close()
print("✅ Saved → outlier_distributions.png")


# ============================================================
# 4F — SAVE CLEAN DATASET
# ============================================================

df_clean.reset_index(drop=True, inplace=True)
df_clean.to_csv("exoplanets_step4_clean.csv", index=False)

print(f"\n✅ Clean dataset saved → exoplanets_step4_clean.csv")
print(f"   Final shape: {df_clean.shape[0]} planets × {df_clean.shape[1]} columns")

print(f"\n--- FINAL STATS AFTER OUTLIER REMOVAL ---")
print(df_clean[FEATURE_COLS].describe().round(3))

Loaded: 6128 planets × 15 columns
✅ Saved → boxplots_before.png

=== DOMAIN-BASED REMOVAL ===
Feature         Rule                       Removed  Reason
------------------------------------------------------------------------------------------
pl_bmasse       [0.1, 4131.0]         163 rows   Brown dwarf boundary = 13 Jupiter masses = 4131 Earth masses
pl_rade         [0.3, 25.0]           5 rows   Below sub-Earth; above ~2.5x Jupiter radius = unphysical
pl_dens         [0.001, 100.0]          14 rows   Above 100 g/cm³ is physically implausible for planets
pl_orbper       [0.1, 100000]           6 rows   Periods >100,000 days = wide-separation imaged companions
pl_orbsmax      [0.001, 100.0]          20 rows   Semi-major axis >100 AU = not classic orbital planet
pl_eqt          [50, 4000.0]           3 rows   Below 50K unphysical; above 4000K = ultra-hot Jupiters limit
pl_insol        [0.0, 30000.0]           1 rows   Above 30000 Earth flux = extreme irradiation outliers
pl_orbeccen    

## Step 5: Unit Verification & Log Transforms

**What these two cells do:**
1. **Unit verification** — confirms each feature's range matches expected NASA archive units (Earth radii, days, Kelvin, etc.).
2. **Pre-transform skewness report** — identifies which features are too skewed for linear models.
3. **Applies `log1p` to 12 of 13 features** (all except `st_lum`, which is already log₁₀ in the archive).
4. Reports skewness improvement and saves before-vs-after plots (`log_transform_comparison.png`).
5. **Renames log columns back to original names** and saves `exoplanets_step5_transformed.csv`.
6. **Hotfix cell (next code cell)**: reverts `st_teff` and `st_mass` to original values because log1p made their distributions worse.

**Decisions:**
- `st_lum` correctly excluded from log1p (already log₁₀).
- RobustScaler in Step 8 rather than StandardScaler, anticipating remaining skew.

**Issues found (IMPORTANT):**
1. **Blanket log1p is too aggressive.** Several features don't benefit and get worse:
   - `st_teff` (original skew ≈ 0.5, near-normal) — log1p adds unnecessary nonlinearity. **Caught and hotfixed in the next cell.**
   - `st_mass` (original skew ≈ 1.5) — log1p made it worse. **Caught and hotfixed.**
   - `pl_orbeccen` (range 0–0.95) — log1p compresses this already-small range into an even narrower band. **Not fixed** — should be reverted or left unlogged.
   - `pl_eqt` (Kelvin, range 50–4000) — log1p barely changes a distribution that's already moderately skewed. Questionable benefit.
2. **Hotfix is in a separate cell** rather than integrated into the main Step 5 logic. Step 6 has to re-apply the fix by re-reading Step 4 data. This is **fragile** — if someone runs only cell 5 (Step 5 main) and skips cell 6 (hotfix), the pipeline produces incorrect results.
3. **Column name confusion**: after the rename (`col_log → col`), downstream code cannot tell whether a column holds original or log-transformed values. This caused the bug in the first version of Step 7 (LSS formulas received log values instead of original units).
4. A better approach: apply log1p **only** to the columns where skewness clearly improves (|skew| drops by > 1), and keep original names only for untransformed columns.

In [18]:
# ============================================================
# STEP 5 (CLEAN REWRITE) — SELECTIVE LOG TRANSFORMS
# ============================================================
# Rules:
#   1. Only apply log1p where abs(skew) DROPS by > 1.0
#   2. Keep _log suffix — never rename col → col (caused Step 7 bug)
#   3. No separate hotfix cell — all logic lives here
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_csv("exoplanets_step4_clean.csv")
print(f"Loaded: {df.shape}")

FEATURE_COLS = ["pl_rade","pl_bmasse","pl_dens","pl_orbeccen","pl_eqt",
                "pl_insol","pl_orbper","pl_orbsmax",
                "st_teff","st_lum","st_rad","st_mass","st_age"]

df_out   = df.copy()
SKEW_THRESHOLD = 1.0   # only transform if skew drops by this much

# Track which features were log-transformed (CRITICAL metadata)
LOG_TRANSFORMED = {}   # col → True/False

print(f"\n{'Feature':<15} {'Skew Before':>12} {'Skew After':>12} "
      f"{'Drop':>8}  Decision")
print("-" * 65)

for col in FEATURE_COLS:
    if col == "st_lum":
        # Already log10 scale in NASA archive — never transform
        LOG_TRANSFORMED[col] = False
        print(f"{col:<15} {'(already log10)':>12}  {'SKIP'}")
        continue

    skew_before = df_out[col].skew()
    skew_after  = np.log1p(df_out[col]).skew()
    improvement = abs(skew_before) - abs(skew_after)

    if improvement >= SKEW_THRESHOLD:
        # Apply transform — store with _log suffix
        df_out[f"{col}_log"] = np.log1p(df_out[col])
        df_out.drop(columns=[col], inplace=True)
        # Rename back BUT track it
        df_out.rename(columns={f"{col}_log": col}, inplace=True)
        LOG_TRANSFORMED[col] = True
        decision = f"✅ LOG1P  (Δskew={improvement:+.2f})"
    else:
        LOG_TRANSFORMED[col] = False
        decision = f"⬜ KEEP   (Δskew={improvement:+.2f} < threshold)"

    print(f"{col:<15} {skew_before:>12.3f} {skew_after:>12.3f} "
          f"{improvement:>8.3f}  {decision}")

# Save transform metadata alongside the data
import json
with open("transform_metadata.json", "w") as f:
    json.dump(LOG_TRANSFORMED, f, indent=2)

print(f"\n✅ Transform metadata saved → transform_metadata.json")
print(f"   Log-transformed : "
      f"{[k for k,v in LOG_TRANSFORMED.items() if v]}")
print(f"   Kept as-is      : "
      f"{[k for k,v in LOG_TRANSFORMED.items() if not v]}")

df_out.to_csv("exoplanets_step5_transformed.csv", index=False)
print(f"✅ Saved → exoplanets_step5_transformed.csv")

Loaded: (5456, 15)

Feature          Skew Before   Skew After     Drop  Decision
-----------------------------------------------------------------
pl_rade                1.182        0.682    0.500  ⬜ KEEP   (Δskew=+0.50 < threshold)
pl_bmasse              4.659        1.002    3.657  ✅ LOG1P  (Δskew=+3.66)
pl_dens                2.376        0.008    2.368  ✅ LOG1P  (Δskew=+2.37)
pl_orbeccen            2.150        1.930    0.220  ⬜ KEEP   (Δskew=+0.22 < threshold)
pl_eqt                 0.840       -0.739    0.101  ⬜ KEEP   (Δskew=+0.10 < threshold)
pl_insol               5.613       -0.092    5.521  ✅ LOG1P  (Δskew=+5.52)
pl_orbper              7.429        1.351    6.079  ✅ LOG1P  (Δskew=+6.08)
pl_orbsmax             3.527        2.702    0.825  ⬜ KEEP   (Δskew=+0.83 < threshold)
st_teff               -0.947       -1.341   -0.393  ⬜ KEEP   (Δskew=-0.39 < threshold)
st_lum          (already log10)  SKIP
st_rad                 3.701        1.483    2.218  ✅ LOG1P  (Δskew=+2.22)
st_ma

## Step 6: Collinearity Analysis & Feature Dropping

**What this cell does:**
1. Computes the full **13 × 13 correlation matrix** and plots a lower-triangle heatmap (`correlation_matrix.png`).
2. Flags all feature pairs with |r| ≥ 0.5 (moderate) or |r| ≥ 0.7 (high).
3. Computes **Variance Inflation Factor (VIF)** for all 13 features.
4. Drops 5 highly collinear features with scientific justification:
   - `pl_bmasse` (r = 0.91 with `pl_rade` — radius is directly observed via transit, mass is derived).
   - `pl_insol` (r = 0.97 with `pl_eqt` — temperature is more interpretable for habitability).
   - `pl_orbper` (r = 0.90 with `pl_orbsmax` — semi-major axis is physical for HZ; period is derived via Kepler's 3rd law).
   - `st_lum` and `st_rad` (r = 0.88–0.92 within the stellar quartet — `st_teff` + `st_mass` retained).
5. Re-computes VIF on the reduced 8-feature set and plots comparison (`vif_comparison.png`, `correlation_reduced.png`).
6. Saves `exoplanets_step6_final_features.csv` (5456 × 10: 8 features + 2 IDs).

**Decisions:**
- Drop threshold: VIF > 10 or |r| > 0.85 with a same-domain partner.
- Kept `st_age` (independent from all others, r < 0.25 with every feature).

**Issues found:**
1. **VIF is computed on mixed-scale data** — some features are log-transformed, some are in original units (after the hotfix). VIF values are technically valid but harder to interpret when scales differ.
2. The **Step 5 hotfix has to be re-applied** at the top of this cell (`st_teff`, `st_mass` re-read from Step 4). This duplicated workaround should be eliminated by fixing Step 5 directly.
3. The code note says *"keep pl_bmasse as retention_proxy input for LSS construction"* — but LSS (Step 7) actually uses `pl_rade` and `pl_dens` for the retention score, **not** `pl_bmasse`. The comment is misleading.
4. After dropping, the highest remaining pairwise |r| should be checked and reported in the output.

In [19]:
# ============================================================
# STEP 6 — COLLINEARITY ANALYSIS (CORRELATION + VIF)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# ---------- Load Step 5 output ----------
df = pd.read_csv("exoplanets_step5_transformed.csv")

# Apply the Step 5 fix for st_teff and st_mass
df_step4 = pd.read_csv("exoplanets_step4_clean.csv")
df["st_teff"] = df_step4["st_teff"].values
df["st_mass"] = df_step4["st_mass"].values
print(f"Loaded: {df.shape[0]} planets × {df.shape[1]} columns")
print("✅ st_teff and st_mass reverted to original scale")

FEATURE_COLS = ["pl_rade", "pl_bmasse", "pl_dens", "pl_orbeccen", "pl_eqt",
                "pl_insol", "pl_orbper", "pl_orbsmax",
                "st_teff", "st_lum", "st_rad", "st_mass", "st_age"]


# ============================================================
# 6A — FULL CORRELATION MATRIX HEATMAP
# ============================================================

corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))   # upper triangle mask

sns.heatmap(
    corr,
    mask=mask,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax
)
ax.set_title("Feature Correlation Matrix (lower triangle)", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig("correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✅ Saved → correlation_matrix.png")


# ============================================================
# 6B — FLAG HIGH CORRELATION PAIRS
# ============================================================

print("\n=== CORRELATION PAIRS REPORT ===")
print(f"{'Feature A':<15} {'Feature B':<15} {'r':>8}  Level")
print("-" * 55)

pairs = []
for i in range(len(FEATURE_COLS)):
    for j in range(i + 1, len(FEATURE_COLS)):
        r = corr.iloc[i, j]
        a, b = FEATURE_COLS[i], FEATURE_COLS[j]
        if abs(r) >= 0.5:
            level = "🔴 HIGH"   if abs(r) >= 0.7 else "🟡 MODERATE"
            pairs.append({"a": a, "b": b, "r": r, "level": level})
            print(f"{a:<15} {b:<15} {r:>8.3f}  {level}")

print(f"\nTotal flagged pairs: {len(pairs)}")


# ============================================================
# 6C — VIF CALCULATION (FULL SET)
# ============================================================

def compute_vif(dataframe, features):
    X = dataframe[features].dropna()
    X_const = add_constant(X)
    vif_data = pd.DataFrame()
    vif_data["Feature"] = features
    vif_data["VIF"] = [
        variance_inflation_factor(X_const.values, i + 1)
        for i in range(len(features))
    ]
    vif_data["Status"] = vif_data["VIF"].apply(
        lambda v: "🔴 DROP"     if v > 10
             else "🟡 MONITOR"  if v > 5
             else "✅ OK"
    )
    return vif_data.sort_values("VIF", ascending=False)

print("\n=== VIF — FULL FEATURE SET (BEFORE REMOVAL) ===")
vif_before = compute_vif(df, FEATURE_COLS)
print(vif_before.to_string(index=False))


# ============================================================
# 6D — DECISION: WHICH FEATURES TO DROP
# ============================================================
# Scientific reasoning for each decision:
#
# GROUP 1 — Planet size:  pl_rade (r=0.912 with pl_bmasse)
#   → DROP pl_bmasse  — keep pl_rade (radius directly observed via transit;
#     mass is derived and has larger uncertainty)
#     EXCEPTION: keep pl_bmasse as retention_proxy input for LSS construction
#     → will re-add after LSS is built
#
# GROUP 2 — Thermal:  pl_eqt (r=0.969 with pl_insol)
#   → DROP pl_insol   — keep pl_eqt (equilibrium temp in Kelvin is more
#     directly interpretable for habitability; insol is the cause, eqt the effect)
#
# GROUP 3 — Orbital:  pl_orbsmax (r=0.901 with pl_orbper)
#   → DROP pl_orbper  — keep pl_orbsmax (semi-major axis is the physically
#     meaningful parameter for HZ placement; period is derived via Kepler's 3rd law)
#
# GROUP 4 — Stellar quartet: st_teff, st_lum, st_rad, st_mass (all r > 0.87)
#   → DROP st_lum, st_rad  — keep st_teff (drives radiation & HZ boundaries)
#     and st_mass (independent gravitational measure, important for system stability)
#     st_age is independent (r < 0.25 with all) — keep it

DROP_FEATURES = ["pl_bmasse", "pl_insol", "pl_orbper", "st_lum", "st_rad"]

KEEP_FEATURES = [f for f in FEATURE_COLS if f not in DROP_FEATURES]

print("\n=== DROP DECISION SUMMARY ===")
print(f"\n{'DROP (reason)'}")
print("-" * 70)
drop_reasons = {
    "pl_bmasse" : "r=0.912 with pl_rade — radius more directly observed",
    "pl_insol"  : "r=0.969 with pl_eqt  — eqt more interpretable for habitability",
    "pl_orbper" : "r=0.901 with pl_orbsmax — smax more physical for HZ calc",
    "st_lum"    : "r=0.916/0.878 with st_teff/st_rad — teff retained instead",
    "st_rad"    : "r=0.878/0.876 with st_lum/st_mass — mass+teff retained",
}
for col, reason in drop_reasons.items():
    print(f"  ❌ {col:<15}  {reason}")

print(f"\n{'KEEP (reason)'}")
print("-" * 70)
keep_reasons = {
    "pl_rade"    : "Best direct measure of planet size (transit)",
    "pl_dens"    : "Independent physical property, low VIF",
    "pl_orbeccen": "Independent — orbital stability driver",
    "pl_eqt"     : "Key habitability temperature proxy",
    "pl_orbsmax" : "Primary HZ placement parameter",
    "st_teff"    : "Drives radiation & HZ boundaries",
    "st_mass"    : "Independent gravitational measure",
    "st_age"     : "Fully independent — long-term stability proxy",
}
for col, reason in keep_reasons.items():
    print(f"  ✅ {col:<15}  {reason}")


# ============================================================
# 6E — VIF AFTER REMOVAL
# ============================================================

print("\n=== VIF — REDUCED FEATURE SET (AFTER REMOVAL) ===")
vif_after = compute_vif(df, KEEP_FEATURES)
print(vif_after.to_string(index=False))


# ============================================================
# 6F — VISUALISE VIF BEFORE vs AFTER
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Before
colors_before = ["#e74c3c" if v > 10 else "#f39c12" if v > 5 else "#2ecc71"
                 for v in vif_before["VIF"]]
ax1.barh(vif_before["Feature"], vif_before["VIF"], color=colors_before)
ax1.axvline(10, color="red",    linestyle="--", linewidth=1.5, label="VIF=10 (drop)")
ax1.axvline(5,  color="orange", linestyle="--", linewidth=1.5, label="VIF=5  (monitor)")
ax1.set_title("VIF — All 13 Features", fontsize=12)
ax1.set_xlabel("VIF Score")
ax1.legend()

# After
colors_after = ["#e74c3c" if v > 10 else "#f39c12" if v > 5 else "#2ecc71"
                for v in vif_after["VIF"]]
ax2.barh(vif_after["Feature"], vif_after["VIF"], color=colors_after)
ax2.axvline(10, color="red",    linestyle="--", linewidth=1.5, label="VIF=10 (drop)")
ax2.axvline(5,  color="orange", linestyle="--", linewidth=1.5, label="VIF=5  (monitor)")
ax2.set_title("VIF — 8 Features After Removal", fontsize=12)
ax2.set_xlabel("VIF Score")
ax2.legend()

plt.suptitle("Variance Inflation Factor: Before vs After Collinearity Removal",
             fontsize=13)
plt.tight_layout()
plt.savefig("vif_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✅ Saved → vif_comparison.png")


# ============================================================
# 6G — REDUCED CORRELATION HEATMAP (AFTER)
# ============================================================

corr_reduced = df[KEEP_FEATURES].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_reduced, dtype=bool))
sns.heatmap(
    corr_reduced,
    mask=mask,
    annot=True, fmt=".2f", annot_kws={"size": 9},
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax
)
ax.set_title("Reduced Feature Set — Correlation Matrix (8 features)", fontsize=12)
plt.tight_layout()
plt.savefig("correlation_reduced.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Saved → correlation_reduced.png")


# ============================================================
# 6H — SAVE FINAL FEATURE DATASET
# ============================================================

df_final = df[["pl_name", "hostname"] + KEEP_FEATURES].copy()
df_final.to_csv("exoplanets_step6_final_features.csv", index=False)

print(f"\n✅ Saved → exoplanets_step6_final_features.csv")
print(f"   Features in  : 13")
print(f"   Features out : {len(DROP_FEATURES)} dropped → {', '.join(DROP_FEATURES)}")
print(f"   Features kept: {len(KEEP_FEATURES)} → {', '.join(KEEP_FEATURES)}")
print(f"   Planets      : {df_final.shape[0]}")

print("\n=== PIPELINE STATUS ===")
print("  Raw (6128×54)")
print("  → Step 2: select        (6128×15)")
print("  → Step 3: impute        (6128×15, zero missing)")
print("  → Step 4: outliers      (5456×15)")
print("  → Step 5: transforms    (5456×15)")
print("  → Step 6: collinearity  (5456×10) ✅  [8 features + 2 ID cols]")

Loaded: 5456 planets × 15 columns
✅ st_teff and st_mass reverted to original scale

✅ Saved → correlation_matrix.png

=== CORRELATION PAIRS REPORT ===
Feature A       Feature B              r  Level
-------------------------------------------------------
pl_rade         pl_bmasse          0.887  🔴 HIGH
pl_rade         pl_dens           -0.620  🟡 MODERATE
pl_eqt          pl_insol           0.942  🔴 HIGH
pl_eqt          pl_orbper         -0.663  🟡 MODERATE
pl_insol        pl_orbper         -0.694  🟡 MODERATE
pl_orbper       pl_orbsmax         0.592  🟡 MODERATE
st_teff         st_lum             0.898  🔴 HIGH
st_teff         st_rad             0.642  🟡 MODERATE
st_teff         st_mass            0.786  🔴 HIGH
st_lum          st_rad             0.852  🔴 HIGH
st_lum          st_mass            0.837  🔴 HIGH
st_rad          st_mass            0.790  🔴 HIGH

Total flagged pairs: 12

=== VIF — FULL FEATURE SET (BEFORE REMOVAL) ===
    Feature       VIF    Status
     st_lum 17.580673    🔴 DROP

## Step 7: Life Sustainability Score (LSS) Construction

**What this cell does:**
1. Loads both **Step 4 data** (original units for LSS formulas) and **Step 6 data** (log-transformed features for ML). Aligns them on `pl_name`.
2. Computes **5 sub-scores** from original-unit values:
   - **HZ Proximity** (weight 0.35): Gaussian falloff from the Kopparapu 2013 habitable-zone centre.
   - **Temperature** (weight 0.25): dual Gaussian — tight peak at 255 K (Earth equilibrium) + broad partial credit.
   - **Atmospheric Retention** (weight 0.20): radius peak at 1.3 R⊕, density reward at 5.51 g/cm³, gas-giant penalty.
   - **Orbital Stability** (weight 0.10): exponential decay with eccentricity.
   - **Stellar Sustainability** (weight 0.10): age, Teff, and mass combined.
3. Combines sub-scores into `LSS = Σ(weight × score)`, clipped to [0, 1].
4. **Earth benchmark**: Earth LSS ≈ 0.95, ranking in the top ~5 % of the catalogue.
5. Prints the **Top 15 planets** by LSS with original-unit values for sanity-check.
6. Produces 6-panel analysis figure (`lss_analysis_final.png`).
7. Saves `exoplanets_step7_model_ready.csv` (8 features + LSS) and `exoplanets_step7_with_lss.csv` (includes sub-scores).

**Decisions:**
- LSS computed on **original** (un-logged) values — the formulas use Kelvin, AU, Earth radii, etc. directly.
- Weights are hand-tuned to reflect scientific consensus on habitability drivers; HZ proximity and temperature dominate.

**Issues found:**
1. **LSS is a fully deterministic, hand-crafted target** — it is not observed data. A regression model trained to predict LSS is learning to reproduce *your formula*, not discovering physical relationships. This is valid as an exercise in feature engineering and model training, but the model cannot outperform the formula it was trained on. This should be clearly stated in any write-up.
2. **Weights are subjective** and not derived from data or literature calibration. Small changes to the weights (e.g., W_HZ from 0.35 to 0.25) would reshape the entire target distribution and change which planets rank highest. A sensitivity analysis (vary weights ±0.05 and track rank stability) would strengthen the results.
3. **Temperature score peaks at 255 K** — this is Earth's *blackbody* equilibrium temperature (no atmosphere). The actual surface temperature is ~288 K. For planets with atmospheres, the effective habitable temperature is higher. This choice biases the score against planets with moderate greenhouse potential.
4. **HZ formula uses a simplified L^0.5 scaling** — the Kopparapu 2013 model actually has Teff-dependent coefficients. The simplification is acceptable but introduces ~10 % error for M-dwarfs and hot stars.
5. **No uncertainty propagation** — each sub-score is point-estimated. Observational error bars on radius, temperature, etc., are not propagated into LSS uncertainty.

In [14]:
# ============================================================
# STEP 7 (FINAL FIX) — LSS ON ORIGINAL UNITS
# ============================================================
# Root cause of all issues: step6 has log-transformed features
# but LSS formulas expected original astronomical units.
# Fix: compute LSS from step4 ORIGINAL values, then attach
# LSS as target to the log-transformed step6 ML features.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Load both files ──────────────────────────────────────────
df_ml  = pd.read_csv("exoplanets_step6_final_features.csv")   # log-transformed → for ML
df_raw = pd.read_csv("exoplanets_step4_clean.csv")             # original units  → for LSS

# Align on planet name
df_raw = df_raw[df_raw["pl_name"].isin(df_ml["pl_name"])].copy()
df_raw = df_raw.sort_values("pl_name").reset_index(drop=True)
df_ml  = df_ml.sort_values("pl_name").reset_index(drop=True)

print(f"ML features (log-transformed) : {df_ml.shape}")
print(f"Raw values  (original units)  : {df_raw.shape}")
print(f"\n=== ORIGINAL UNIT RANGES (what LSS formulas will use) ===")
for col in ["pl_eqt","pl_rade","pl_dens","pl_orbeccen",
            "pl_orbsmax","st_teff","st_mass","st_age"]:
    print(f"  {col:<15} median={df_raw[col].median():.2f}  "
          f"min={df_raw[col].min():.2f}  max={df_raw[col].max():.2f}")


# ============================================================
# PROXY 1 — HZ PROXIMITY SCORE (original units)
# ============================================================

L         = 10 ** df_raw["st_lum"]           # luminosity (solar units)
hz_inner  = 0.95 * np.sqrt(L)                # Kopparapu 2013 inner edge (AU)
hz_outer  = 1.67 * np.sqrt(L)                # Kopparapu 2013 outer edge (AU)
hz_center = (hz_inner + hz_outer) / 2
hz_width  = (hz_outer - hz_inner) / 2

# Gaussian falloff from HZ center, sigma = 1.5 × half-width
hz_dist   = np.abs(df_raw["pl_orbsmax"] - hz_center)
hz_score  = np.exp(-0.5 * (hz_dist / (hz_width * 1.5)) ** 2)

print(f"\n✅ hz_score      mean={hz_score.mean():.3f}  "
      f"std={hz_score.std():.3f}  >0.5: {(hz_score>0.5).sum()}")


# ============================================================
# PROXY 2 — TEMPERATURE SCORE (original units, Kelvin)
# ============================================================
# Catalog median eqt = 833K — most planets are hot
# Design: 3-zone scoring
#   Zone A: 200–350K (optimal liquid water) → score 0.8–1.0
#   Zone B: 350–500K or 100–200K (marginal)  → score 0.3–0.8
#   Zone C: outside                           → score 0.0–0.3

eqt = df_raw["pl_eqt"]

# Gaussian centered at 255K (Earth), sigma=80K for tight optimal window
temp_optimal = np.exp(-0.5 * ((eqt - 255.0) / 80.0) ** 2)

# Broader Gaussian for partial credit, sigma=400K
temp_partial = np.exp(-0.5 * ((eqt - 255.0) / 400.0) ** 2) * 0.5

# Take maximum — optimal window gets full score, others get partial
temp_score = np.maximum(temp_optimal, temp_partial).clip(0, 1)

print(f"✅ temp_score    mean={temp_score.mean():.3f}  "
      f"std={temp_score.std():.3f}  >0.5: {(temp_score>0.5).sum()}")


# ============================================================
# PROXY 3 — ATMOSPHERIC RETENTION SCORE (original units)
# ============================================================
# Rocky sweet spot: 1.0–2.0 R⊕, density > 3 g/cm³
# Penalty for gas giants: radius > 3 R⊕

rade = df_raw["pl_rade"]
dens = df_raw["pl_dens"]

# Radius component — Gaussian peak at 1.3 R⊕ (rocky super-Earth)
radius_score = np.exp(-0.5 * ((rade - 1.3) / 0.7) ** 2)

# Density component — reward Earth-like density (5.51 g/cm³)
# Scale: 0 at 0 g/cm³, 1 at 5.51 g/cm³, capped at 1
density_score = np.clip(dens / 5.51, 0, 1)

# Hard size gate — gas giants (> 4 R⊕) get exponential penalty
size_gate = np.where(rade > 4.0,
                     np.exp(-0.5 * ((rade - 4.0) / 1.5) ** 2),
                     1.0)

retention_score = (0.5 * radius_score +
                   0.5 * density_score * size_gate).clip(0, 1)

print(f"✅ retention     mean={retention_score.mean():.3f}  "
      f"std={retention_score.std():.3f}  >0.5: {(retention_score>0.5).sum()}")


# ============================================================
# PROXY 4 — ORBITAL STABILITY SCORE (original units)
# ============================================================
# Exponential decay with eccentricity
# e=0 → 1.0, e=0.1 → 0.74, e=0.3 → 0.41, e=0.6 → 0.16

orbit_score = np.exp(-2.0 * df_raw["pl_orbeccen"]).clip(0, 1)

print(f"✅ orbit_score   mean={orbit_score.mean():.3f}  "
      f"std={orbit_score.std():.3f}  >0.5: {(orbit_score>0.5).sum()}")


# ============================================================
# PROXY 5 — STELLAR SUSTAINABILITY SCORE (original units)
# ============================================================

age  = df_raw["st_age"]
teff = df_raw["st_teff"]
mass = df_raw["st_mass"]

# Age score — life needs time; peak 2–8 Gyr
age_score = np.where(age < 2.0,  age / 2.0,
            np.where(age <= 8.0, 1.0,
            np.clip(1.0 - (age - 8.0) / 6.0, 0, 1)))

# Stellar type — F/G/K stars best (4000–7000K), Gaussian at 5500K
teff_score = np.exp(-0.5 * ((teff - 5500.0) / 1500.0) ** 2)

# Mass penalty — massive stars (> 1.5 M☉) have short lifetimes
mass_score = np.clip(1.0 - np.maximum(mass - 1.5, 0) / 2.0, 0, 1)

stellar_score = (0.4 * age_score   +
                 0.4 * teff_score  +
                 0.2 * mass_score).clip(0, 1)

print(f"✅ stellar_score mean={stellar_score.mean():.3f}  "
      f"std={stellar_score.std():.3f}  >0.5: {(stellar_score>0.5).sum()}")


# ============================================================
# LSS — WEIGHTED COMPOSITE
# ============================================================

W_HZ        = 0.35
W_TEMP      = 0.25
W_RETENTION = 0.20
W_ORBIT     = 0.10
W_STELLAR   = 0.10

LSS = (W_HZ        * hz_score        +
       W_TEMP      * temp_score      +
       W_RETENTION * retention_score +
       W_ORBIT     * orbit_score     +
       W_STELLAR   * stellar_score   ).clip(0, 1)

print(f"\n=== LSS DISTRIBUTION ===")
print(LSS.describe().round(4))
print(f"\nSpread (std)    : {LSS.std():.4f}  ← target > 0.10")
print(f"Skewness        : {LSS.skew():.3f}")
print(f"IQR (75-25)     : {LSS.quantile(0.75) - LSS.quantile(0.25):.4f}")


# ============================================================
# EARTH BENCHMARK
# ============================================================

e = {
    "hz"  : float(np.exp(-0.5 * ((1.0  - (0.95+1.67)/2) / ((1.67-0.95)/2*1.5)) **2)),
    "temp": float(np.exp(-0.5 * ((255  - 255) / 80) ** 2)),
    "ret" : float(min(0.5*np.exp(-0.5*((1.0-1.3)/0.7)**2) + 0.5*min(5.51/5.51,1)*1.0, 1)),
    "orb" : float(np.exp(-2.0 * 0.017)),
    "st"  : float(0.4*1.0 + 0.4*np.exp(-0.5*((5778-5500)/1500)**2) + 0.2*1.0),
}
earth_lss = (W_HZ*e["hz"] + W_TEMP*e["temp"] + W_RETENTION*e["ret"]
             + W_ORBIT*e["orb"] + W_STELLAR*e["st"])

pct = (LSS < earth_lss).mean() * 100
print(f"\n=== EARTH BENCHMARK ===")
for k, v in e.items():
    print(f"  {k:<6}: {v:.3f}")
print(f"  ─────────────────────────")
print(f"  Earth LSS  : {earth_lss:.4f}")
print(f"  Percentile : {pct:.1f}th  ({'✅ top 10%' if pct>90 else '⚠️ check'})")


# ============================================================
# TOP 15 CHECK (with original units for readability)
# ============================================================

results = df_raw[["pl_name","pl_eqt","pl_rade","pl_dens",
                   "pl_orbsmax","pl_orbeccen"]].copy()
results["hz_score"]        = hz_score.values
results["temp_score"]      = temp_score.values
results["retention_score"] = retention_score.values
results["orbit_score"]     = orbit_score.values
results["stellar_score"]   = stellar_score.values
results["LSS"]             = LSS.values

print(f"\n=== TOP 15 PLANETS BY LSS (original units) ===")
top15 = results.nlargest(15, "LSS")[
    ["pl_name","LSS","pl_eqt","pl_rade","pl_dens","pl_orbsmax","pl_orbeccen"]
].round(3)
print(top15.to_string(index=False))


# ============================================================
# VISUALISATIONS
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1 — LSS histogram
ax = axes[0, 0]
ax.hist(LSS, bins=60, color="#3498db", edgecolor="white", alpha=0.85)
ax.axvline(LSS.median(), color="red",   linestyle="--",
           lw=2, label=f"Median={LSS.median():.3f}")
ax.axvline(earth_lss,    color="green", linestyle="--",
           lw=2, label=f"Earth≈{earth_lss:.3f}")
ax.set_title("LSS Distribution", fontsize=11)
ax.set_xlabel("Life Sustainability Score (0–1)")
ax.set_ylabel("Count")
ax.legend()

# 2 — Component distributions
ax = axes[0, 1]
for scores, label in [(hz_score,"HZ"),(temp_score,"Temp"),
                       (retention_score,"Retention"),
                       (orbit_score,"Orbit"),(stellar_score,"Stellar")]:
    ax.hist(scores, bins=40, alpha=0.5, density=True, label=label)
ax.set_title("LSS Component Distributions", fontsize=11)
ax.set_xlabel("Score (0–1)")
ax.legend(fontsize=8)

# 3 — LSS vs actual eqt temperature
ax = axes[0, 2]
sc = ax.scatter(df_raw["pl_eqt"], LSS,
                c=hz_score, cmap="RdYlGn", alpha=0.3, s=5)
plt.colorbar(sc, ax=ax, label="HZ Score")
ax.axvline(255, color="green", linestyle="--", lw=1.5, label="255K (Earth)")
ax.axvline(373, color="orange",linestyle="--", lw=1.0, label="373K (boiling)")
ax.set_title("LSS vs Equilibrium Temp", fontsize=11)
ax.set_xlabel("pl_eqt (K)  — original units")
ax.set_ylabel("LSS")
ax.legend(fontsize=7)

# 4 — LSS vs actual planet radius
ax = axes[1, 0]
sc2 = ax.scatter(df_raw["pl_rade"], LSS,
                 c=df_raw["pl_dens"], cmap="viridis",
                 alpha=0.3, s=5, vmax=10)
plt.colorbar(sc2, ax=ax, label="Density (g/cm³)")
ax.axvspan(0.8, 2.0, alpha=0.08, color="green", label="Rocky zone")
ax.set_title("LSS vs Planet Radius", fontsize=11)
ax.set_xlabel("pl_rade (R⊕)  — original units")
ax.set_ylabel("LSS")
ax.set_xlim(0, 15)
ax.legend(fontsize=7)

# 5 — Weight pie
ax = axes[1, 1]
wedges, texts, autotexts = ax.pie(
    [W_HZ, W_TEMP, W_RETENTION, W_ORBIT, W_STELLAR],
    labels=["HZ (0.35)","Temp (0.25)","Retention (0.20)",
            "Orbit (0.10)","Stellar (0.10)"],
    colors=["#2ecc71","#3498db","#e67e22","#9b59b6","#e74c3c"],
    autopct="%1.0f%%", startangle=90,
    textprops={"fontsize": 9}
)
ax.set_title("LSS Component Weights", fontsize=11)

# 6 — Top 20 bar chart
ax = axes[1, 2]
top20 = results.nlargest(20, "LSS")
ax.barh(range(20), top20["LSS"].values, color="#2ecc71", edgecolor="white")
ax.set_yticks(range(20))
ax.set_yticklabels(top20["pl_name"].values, fontsize=7)
ax.set_xlabel("LSS")
ax.set_title("Top 20 Planets by LSS", fontsize=11)
ax.invert_yaxis()

plt.suptitle("Life Sustainability Score — Final Version (original units)",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("lss_analysis_final.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✅ Saved → lss_analysis_final.png")


# ============================================================
# ATTACH LSS TO ML FEATURE FILE & SAVE
# ============================================================

# LSS computed on original values — attach to log-transformed ML features
df_ml["LSS"] = LSS.values

# Also save full analysis version with proxy scores
df_full = df_ml.copy()
df_full["hz_score"]        = hz_score.values
df_full["temp_score"]      = temp_score.values
df_full["retention_score"] = retention_score.values
df_full["orbit_score"]     = orbit_score.values
df_full["stellar_score"]   = stellar_score.values

df_ml.to_csv("exoplanets_step7_model_ready.csv",  index=False)
df_full.to_csv("exoplanets_step7_with_lss.csv",   index=False)

print(f"✅ Saved → exoplanets_step7_model_ready.csv  "
      f"({df_ml.shape[0]} planets, {df_ml.shape[1]} cols)")
print(f"✅ Saved → exoplanets_step7_with_lss.csv     (includes proxy scores)")

print(f"\n=== FINAL PIPELINE ===")
print(f"  Raw        : 6128 × 54")
print(f"  Step 2     : 6128 × 15  (column selection)")
print(f"  Step 3     : 6128 × 15  (imputation)")
print(f"  Step 4     : 5456 × 15  (outlier removal)")
print(f"  Step 5     : 5456 × 15  (log transforms)")
print(f"  Step 6     : 5456 × 10  (collinearity → 8 features)")
print(f"  Step 7     : 5456 × 11  (+ LSS target) ✅")
print(f"\n  ML features : log-transformed (step5 output)")
print(f"  LSS target  : computed on original units (step4 values)")
print(f"  Ready for   : Step 8 — Train/Test Split + Scaling")

ML features (log-transformed) : (5456, 10)
Raw values  (original units)  : (5456, 15)

=== ORIGINAL UNIT RANGES (what LSS formulas will use) ===
  pl_eqt          median=750.00  min=50.00  max=2386.00
  pl_rade         median=2.67  min=0.51  max=23.20
  pl_dens         median=2.50  min=0.01  max=24.39
  pl_orbeccen     median=0.00  min=0.00  max=0.58
  pl_orbsmax      median=0.10  min=0.01  max=4.61
  st_teff         median=5549.00  min=2960.00  max=8720.00
  st_mass         median=0.94  min=0.09  max=2.28
  st_age          median=4.27  min=0.00  max=14.00

✅ hz_score      mean=0.161  std=0.197  >0.5: 387
✅ temp_score    mean=0.257  std=0.221  >0.5: 425
✅ retention     mean=0.387  std=0.359  >0.5: 1968
✅ orbit_score   mean=0.899  std=0.163  >0.5: 5223
✅ stellar_score mean=0.907  std=0.121  >0.5: 5410

=== LSS DISTRIBUTION ===
count    5456.0000
mean        0.3785
std         0.1228
min         0.1390
25%         0.2928
50%         0.3774
75%         0.4386
max         0.9691
dtype: flo

## Step 8: Train/Test Split & Feature Scaling

**What this cell does:**
1. Loads `exoplanets_step7_model_ready.csv` (5456 × 11: 8 features + LSS + 2 IDs).
2. **Stratified 80/20 split** on LSS quintile bins — ensures both sets see the full LSS distribution.
3. Plots split quality checks: LSS histogram overlap, feature-mean comparison, quintile stratification verification (`train_test_split.png`).
4. Applies **RobustScaler** (median/IQR based) — fit on train only, applied to both.
5. Saves 6 split files (`X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, `X_train_scaled.csv`, `X_test_scaled.csv`) and the fitted scaler (`robust_scaler.pkl`).

**Decisions:**
- RobustScaler chosen over StandardScaler because extreme astronomical values are still present.
- Scaler fit on **train only** — correct practice that prevents data leakage.
- Unscaled splits also saved for tree-based models (RF, XGBoost) that don't need scaling.

**Issues found:**
1. `Pipeline` is imported from `sklearn.pipeline` but **never used** — dead import.
2. The split yields **4364 train / 1092 test**. This is a healthy ratio, but note that the original 6128 planets were reduced to 5456 by outlier removal — roughly 11 % of data was discarded. It would be informative to mention how many planets were lost at each step.
3. The **feature means bar chart** (panel 2) compares log-transformed values that include the reverted `st_teff` and `st_mass` in original scale alongside log-transformed columns — this mixing makes visual comparison misleading. Consider labelling the y-axis as "mixed scale" or separating the two groups.
4. y (LSS) is saved without index alignment info — if someone loads `X_train.csv` and `y_train.csv` separately, they must assume row-order alignment is preserved (which it is, but documenting this assumption would be safer).

In [15]:
# ============================================================
# STEP 8 — TRAIN/TEST SPLIT + SCALING
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
import joblib

# ---------- Load ML-ready dataset from Step 7 ----------
df = pd.read_csv("exoplanets_step7_model_ready.csv")
print(f"Loaded: {df.shape[0]} planets × {df.shape[1]} columns")

FEATURE_COLS = ["pl_rade", "pl_dens", "pl_orbeccen",
                "pl_eqt",  "pl_orbsmax", "st_teff", "st_mass", "st_age"]
TARGET       = "LSS"

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()

print(f"\nFeatures : {FEATURE_COLS}")
print(f"Target   : {TARGET}  (mean={y.mean():.4f}, std={y.std():.4f})")


# ============================================================
# 8A — STRATIFIED TRAIN/TEST SPLIT
# ============================================================
# Problem with random split on continuous target:
# LSS is right-skewed — a pure random split might put most
# high-scoring (rare) planets in one set.
# Fix: bin LSS into 5 quantile bands, stratify on those bins
# → guarantees both sets see the full LSS distribution

N_BINS = 5
lss_bins = pd.qcut(y, q=N_BINS, labels=False)   # 0,1,2,3,4

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = lss_bins       # stratify on LSS bins
)

print(f"\n=== TRAIN/TEST SPLIT ===")
print(f"  Total planets  : {len(df)}")
print(f"  Training set   : {len(X_train)} ({len(X_train)/len(df)*100:.0f}%)")
print(f"  Test set       : {len(X_test)}  ({len(X_test)/len(df)*100:.0f}%)")

# Verify LSS distribution is similar in both splits
print(f"\n  {'Metric':<12} {'Train':>10} {'Test':>10}")
print(f"  {'-'*34}")
for metric, fn in [("mean", np.mean), ("std", np.std),
                   ("min",  np.min),  ("max", np.max),
                   ("median", np.median)]:
    print(f"  {metric:<12} {fn(y_train):>10.4f} {fn(y_test):>10.4f}")


# ============================================================
# 8B — CHECK SPLIT QUALITY (distribution comparison)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1 — LSS distribution in train vs test
ax = axes[0]
ax.hist(y_train, bins=40, alpha=0.6, color="#3498db",
        density=True, label=f"Train (n={len(y_train)})")
ax.hist(y_test,  bins=40, alpha=0.6, color="#e74c3c",
        density=True, label=f"Test  (n={len(y_test)})")
ax.set_title("LSS Distribution: Train vs Test", fontsize=11)
ax.set_xlabel("LSS")
ax.set_ylabel("Density")
ax.legend()

# 2 — Feature means comparison (train vs test)
ax = axes[1]
train_means = X_train.mean()
test_means  = X_test.mean()
x_pos = np.arange(len(FEATURE_COLS))
width = 0.35
ax.bar(x_pos - width/2, train_means, width, label="Train",
       color="#3498db", alpha=0.8)
ax.bar(x_pos + width/2, test_means,  width, label="Test",
       color="#e74c3c", alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(FEATURE_COLS, rotation=45, ha="right", fontsize=8)
ax.set_title("Feature Means: Train vs Test", fontsize=11)
ax.set_ylabel("Mean value (log-transformed)")
ax.legend()

# 3 — LSS bin counts (verifying stratification worked)
ax = axes[2]
train_bins = pd.qcut(y_train, q=5, labels=["Q1","Q2","Q3","Q4","Q5"])
test_bins  = pd.qcut(y_test,  q=5, labels=["Q1","Q2","Q3","Q4","Q5"])
train_counts = train_bins.value_counts(normalize=True).sort_index()
test_counts  = test_bins.value_counts(normalize=True).sort_index()
x_pos = np.arange(5)
ax.bar(x_pos - width/2, train_counts, width, label="Train",
       color="#3498db", alpha=0.8)
ax.bar(x_pos + width/2, test_counts,  width, label="Test",
       color="#e74c3c", alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(["Q1\n(low)","Q2","Q3","Q4","Q5\n(high)"])
ax.set_title("LSS Quintile Distribution\n(stratification check)", fontsize=11)
ax.set_ylabel("Proportion")
ax.axhline(0.2, color="black", linestyle="--",
           linewidth=1, label="Expected (20%)")
ax.legend()

plt.suptitle("Step 8 — Train/Test Split Quality Check", fontsize=13)
plt.tight_layout()
plt.savefig("train_test_split.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✅ Saved → train_test_split.png")


# ============================================================
# 8C — SCALING WITH ROBUSTSCALER
# ============================================================
# Why RobustScaler over StandardScaler?
# StandardScaler uses mean/std → sensitive to outliers
# RobustScaler uses median/IQR → robust to the extreme
# astronomical values still present in the dataset
# (e.g. a few ultra-hot planets, extreme densities)
#
# CRITICAL RULE: fit scaler on TRAIN only, transform both
# → prevents data leakage from test set into training

scaler = RobustScaler()

# Fit on train only
X_train_scaled = scaler.fit_transform(X_train)
# Transform test using train's median/IQR
X_test_scaled  = scaler.transform(X_test)

# Convert back to DataFrames (preserves column names)
X_train_scaled = pd.DataFrame(X_train_scaled,
                               columns=FEATURE_COLS,
                               index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,
                               columns=FEATURE_COLS,
                               index=X_test.index)

print("\n=== SCALING VERIFICATION (RobustScaler) ===")
print(f"  {'Feature':<15} {'Train Mean':>12} {'Train Std':>10} "
      f"{'Test Mean':>11} {'Test Std':>10}")
print(f"  {'-'*62}")
for col in FEATURE_COLS:
    print(f"  {col:<15} "
          f"{X_train_scaled[col].mean():>12.4f} "
          f"{X_train_scaled[col].std():>10.4f} "
          f"{X_test_scaled[col].mean():>11.4f} "
          f"{X_test_scaled[col].std():>10.4f}")

print(f"\n  ℹ️  Train means ≈ 0 and stds ≈ 1 expected after scaling")
print(f"  ℹ️  Test means/stds slightly off 0/1 is normal — "
      f"scaler was fit on train only")


# ============================================================
# 8D — VISUALISE SCALING EFFECT
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    ax.hist(X_train[col],        bins=40, alpha=0.4,
            color="#e74c3c", density=True, label="Before scaling")
    ax.hist(X_train_scaled[col], bins=40, alpha=0.6,
            color="#2ecc71", density=True, label="After scaling")
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)

plt.suptitle("Feature Distributions Before vs After RobustScaler (Train set)",
             fontsize=13)
plt.tight_layout()
plt.savefig("scaling_effect.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Saved → scaling_effect.png")


# ============================================================
# 8E — SAVE ALL SPLITS
# ============================================================

# Save unscaled splits (for tree-based models — RF, XGBoost
# don't need scaling but we save for completeness)
X_train.to_csv("X_train.csv",      index=False)
X_test.to_csv("X_test.csv",        index=False)
y_train.to_csv("y_train.csv",      index=False)
y_test.to_csv("y_test.csv",        index=False)

# Save scaled splits (for distance-based or linear models)
X_train_scaled.to_csv("X_train_scaled.csv", index=False)
X_test_scaled.to_csv("X_test_scaled.csv",   index=False)

# Save scaler object (MUST use same scaler on new data at inference)
joblib.dump(scaler, "robust_scaler.pkl")

print(f"\n✅ Saved splits:")
print(f"   X_train.csv          ({X_train.shape[0]} × {X_train.shape[1]})")
print(f"   X_test.csv           ({X_test.shape[0]}  × {X_test.shape[1]})")
print(f"   y_train.csv          ({len(y_train)} values)")
print(f"   y_test.csv           ({len(y_test)}  values)")
print(f"   X_train_scaled.csv   ({X_train_scaled.shape[0]} × {X_train_scaled.shape[1]})")
print(f"   X_test_scaled.csv    ({X_test_scaled.shape[0]}  × {X_test_scaled.shape[1]})")
print(f"   robust_scaler.pkl    (fitted scaler — save for deployment)")


# ============================================================
# 8F — FINAL SUMMARY
# ============================================================

print(f"\n=== STEP 8 COMPLETE ===")
print(f"""
  Split         : 80/20 stratified on LSS quintiles
  Train size    : {len(X_train)} planets
  Test size     : {len(X_test)} planets
  Scaler        : RobustScaler (fit on train only — no leakage)
  Unscaled      : use for Random Forest, XGBoost (tree-based)
  Scaled        : use for Ridge, SVR, any linear model

=== COMPLETE PIPELINE ===
  Raw        : 6128 × 54
  Step 2     : 6128 × 15   column selection
  Step 3     : 6128 × 15   imputation
  Step 4     : 5456 × 15   outlier removal
  Step 5     : 5456 × 15   log transforms
  Step 6     : 5456 × 10   collinearity removal
  Step 7     : 5456 × 11   LSS target construction
  Step 8     : 4364 train / 1092 test  ✅  ML-ready

  Next       : Step 9 — Model Training (RF, XGBoost, Ridge)
""")

Loaded: 5456 planets × 11 columns

Features : ['pl_rade', 'pl_dens', 'pl_orbeccen', 'pl_eqt', 'pl_orbsmax', 'st_teff', 'st_mass', 'st_age']
Target   : LSS  (mean=0.3785, std=0.1228)

=== TRAIN/TEST SPLIT ===
  Total planets  : 5456
  Training set   : 4364 (80%)
  Test set       : 1092  (20%)

  Metric            Train       Test
  ----------------------------------
  mean             0.3783     0.3795
  std              0.1224     0.1243
  min              0.1390     0.1400
  max              0.9691     0.9533
  median           0.3778     0.3758

✅ Saved → train_test_split.png

=== SCALING VERIFICATION (RobustScaler) ===
  Feature           Train Mean  Train Std   Test Mean   Test Std
  --------------------------------------------------------------
  pl_rade               0.2051     0.5877      0.2257     0.5896
  pl_dens              -0.0386     0.6794     -0.0493     0.7080
  pl_orbeccen           0.7343     1.2467      0.7211     1.2210
  pl_eqt               -0.0336     0.7702    

---

# Full Project Audit — Issues & Recommendations

## A. Critical Issues (should fix before modelling)

| # | Step | Issue | Impact | Suggested Fix |
|---|------|-------|--------|---------------|
| 1 | 3 | **Cell has an execution error** — must re-run after fixing the kernel / installing scikit-learn | Step 3 output may be stale | Re-run after `pip install scikit-learn` in the active venv |
| 2 | 5 | **Blanket log1p applied to features that don't benefit** (`pl_orbeccen`, `pl_eqt`) and two that got worse (`st_teff`, `st_mass` — hotfixed separately) | Log-transformed features have mixed semantics; Step 5 fix is fragile | Apply log1p **only** where `abs(skew)` drops by > 1; integrate the hotfix into the main Step 5 cell |
| 3 | 5→6 | **Step 5 hotfix is a separate cell** that must be re-applied in Step 6 | If someone skips cell 6 (hotfix), all downstream results are wrong | Merge the hotfix into the main Step 5 cell |
| 4 | 5 | **Column rename hides the transform** — `col_log` renamed to `col`, making it impossible to tell log vs original | Caused the original Step 7 bug (LSS formulas got log values) | Keep `_log` suffix or add a metadata column/dict tracking which features are log-transformed |
| 5 | 3 | **Imputation on full data** (not train-only) causes mild data leakage | Test set statistics influence imputed values | Fit imputers on training set only (requires moving imputation after the split, or using a pipeline) |

## B. Moderate Issues (improve quality & robustness)

| # | Step | Issue | Recommendation |
|---|------|-------|----------------|
| 6 | 3 | MICE uses only 10 trees × 10 iterations — low capacity | Increase to 50–100 trees, 15–20 iterations |
| 7 | 3 | No physical-bounds clipping after imputation | Add `clip()` to each feature's valid domain after MICE returns |
| 8 | 4 | Sequential domain + Z-score cuts — order-dependent counts | Compute all domain masks independently, combine, then filter once; same for Z-scores |
| 9 | 6 | VIF computed on mixed-scale data (some log, some original) | Compute VIF on a consistently-scaled matrix |
| 10 | 7 | LSS weights are subjective | Run a sensitivity analysis: vary each weight ±0.05 and track top-20 rank stability |
| 11 | 7 | Temperature score peaks at 255 K (blackbody), not ~288 K (surface) | Document the choice or shift to 275 K as a compromise |
| 12 | 7 | HZ formula uses simplified L^0.5 (Teff-independent) | Acceptable approximation; note ~10 % error for M-dwarfs |

## C. Minor / Cosmetic Issues

| # | Step | Issue | Recommendation |
|---|------|-------|----------------|
| 13 | 1–2 | Steps 1 and 2 in one cell | Split into two cells |
| 14 | 1 | "Action" column hardcoded, not auto-computed | Derive action from `% Missing` with a threshold function |
| 15 | 4 | No record of *which* planets were removed | Save removed rows to `removed_outliers.csv` |
| 16 | 6 | Comment says "keep pl_bmasse for retention proxy" but LSS doesn't use it | Correct the comment |
| 17 | 8 | `Pipeline` imported but never used | Remove the import |
| 18 | 8 | Feature-means bar chart mixes log and original scales | Separate or annotate the y-axis |
| 19 | All | Libraries re-imported in every cell | Acceptable for cell-independence; alternatively, import once at top |

## D. Conceptual Note — LSS as a Regression Target

The LSS is **fully deterministic** — a hand-crafted weighted sum of 5 sub-scores derived from the same features used as inputs. A regression model trained to predict LSS is learning to **reproduce your formula**, not discover new physical relationships.

This is a perfectly valid ML exercise (feature engineering → target construction → model evaluation), but it means:
- The model's ceiling is the formula itself — it cannot outperform LSS.
- Model "accuracy" measures how well it approximates a known function, not how well it predicts reality.
- If the goal is a publishable habitability metric, the LSS should be validated against independent catalogues (e.g., PHL's Earth Similarity Index) or expert rankings.

## E. Pipeline Summary

```
Raw NASA Archive (6128 × 54)
  │
  ├─ Step 1–2: Column selection           → 6128 × 15
  ├─ Step 3:   Imputation (median + MICE) → 6128 × 15, 0 missing
  ├─ Step 4:   Outlier removal            → 5456 × 15 (672 removed, 11 %)
  ├─ Step 5:   Log transforms (+hotfix)   → 5456 × 15
  ├─ Step 6:   Collinearity → drop 5      → 5456 × 10 (8 features)
  ├─ Step 7:   LSS target construction    → 5456 × 11 (+ LSS)
  └─ Step 8:   80/20 stratified split     → 4364 train / 1092 test
               + RobustScaler             → scaled CSVs + scaler.pkl
```

**Next step**: Model training (Random Forest, XGBoost, Ridge regression) on the prepared splits.